# 📄 Descripción general del código

 Este script está diseñado para ejecutarse en Google Colab y preparar datos de entrenamiento

 para un modelo de clasificación de oraciones usando RoBERTa en español.

 A partir de múltiples archivos Excel con anotaciones de fragmentos (CLAIM, EVIDENCE, OTHER),

 el script realiza los siguientes pasos:

 1. Carga y combina los archivos .xlsx desde Google Drive.
 2. Divide el texto original en oraciones usando puntuación básica.
 3. Asocia cada oración con una etiqueta según su posición en el texto original.
 4. Expande el contexto de cada oración incluyendo ±4 oraciones vecinas.
 5. Tokeniza los bloques de texto usando el tokenizador de RoBERTa.
 6. Guarda el dataset final en formato `.jsonl`, compatible con Hugging Face Trainer.

 El resultado es adecuado para entrenar un modelo multiclase (CLAIM/EVIDENCE/OTHER).

In [ ]:
!pip install transformers jsonlines

In [ ]:
import pandas as pd
import jsonlines
import re
import os
from transformers import AutoTokenizer
from google.colab import drive

In [ ]:
# 🚀 Montar Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ✅ Definir ruta de la carpeta que contiene los archivos .xlsx

folder_path = "data/0318_SpaCy 데이터 생성 전 수작업 필요한 폴더/final"
file_list = [f for f in os.listdir(folder_path) if f.endswith(".xlsx")]

In [ ]:
# ✅ Leer y combinar todos los archivos Excel

all_dfs = []
for filename in file_list:
    file_path = os.path.join(folder_path, filename)
    df = pd.read_excel(file_path)
    df = df[['answer_id', 'label', 'Extracted Text', 'Start Index', 'End Index', 'Original Text']]
    df = df.dropna(subset=['Original Text'])
    all_dfs.append(df)

df_all = pd.concat(all_dfs, ignore_index=True)

In [ ]:
# Función simple para separar oraciones en español
def simple_spanish_sentence_split(text):
    text = re.sub(r'\s+', ' ', text.strip())
    return re.split(r'(?<=[.!?]) +', text)

In [ ]:
# Asignar etiquetas a cada oración
label_map = {'CLAIM': 0, 'EVIDENCE': 1, 'OTHER': 2}
labeled_sentences = []

grouped = df.groupby("answer_id").first().reset_index()
for _, row in grouped.iterrows():
    rid = row['answer_id']
    original_text = row['Original Text']
    sentences = simple_spanish_sentence_split(original_text)
    spans = df[df['answer_id'] == rid]

    char_index = 0
    for i, sent in enumerate(sentences):
        start = original_text.find(sent, char_index)
        end = start + len(sent)
        char_index = end

        match = spans[(spans['Start Index'] < end) & (spans['End Index'] > start)]
        label = match.iloc[0]['label'] if not match.empty else 'OTHER'

        labeled_sentences.append({
            'answer_id': rid,
            'sentence_idx': i,
            'sentence': sent,
            'label': label
        })

In [ ]:
# Incluir contexto: ±4 oraciones vecinas
context_size = 4
contextualized = []
for row in labeled_sentences:
    rid = row['answer_id']
    idx = row['sentence_idx']
    context_block = [r['sentence'] for r in labeled_sentences
                     if r['answer_id'] == rid and abs(r['sentence_idx'] - idx) <= context_size]
    full_text = ' '.join(context_block)
    contextualized.append({
        'text': full_text,
        'label': label_map[row['label']]
    })

In [ ]:
# Tokenizador RoBERTa
tokenizer = AutoTokenizer.from_pretrained("PlanTL-GOB-ES/roberta-base-bne")

In [ ]:
# Tokenizar texto y preparar dataset final

dataset = []
for row in contextualized:
    encoding = tokenizer(
        row['text'],
        padding="max_length",
        truncation=True,
        max_length=512
    )
    dataset.append({
        "input_ids": encoding['input_ids'],
        "attention_mask": encoding['attention_mask'],
        "labels": row['label']
    })

In [ ]:
#Crear carpeta de salida si no existe

import os

output_dir = "data"

# Crear carpeta de salida si no existe
os.makedirs(output_dir, exist_ok=True)

# Guardar en archivo .jsonl
output_path = os.path.join(output_dir, "sw4_roberta_dataset_all.jsonl")

with jsonlines.open(output_path, mode="w") as writer:
    for item in dataset:
        writer.write(item)

print(f"✅ Saved: {output_path}")
# print(f"✅ Guardado completo: {output_path}")

✅ Saved: data/claim_evidence_train.jsonl
